In [0]:
import numpy as np
import pandas as pd
import os
pd.set_option('display.max_rows', None)
#Directory and filename--BBH1s,BBH4s,BBH10s and Noise
directories=['/home/shared/MDC/new/BBH_1_v2','/home/shared/MDC/new/BBH_4_v2','/home/shared/MDC/new/BBH_10_v2','/home/shared/MDC/long_data/noise']

for k in range(0,1):
    directory=directories[k]
    for i in range(0,100):
        print('file:'+str(i))
        fileserie=directory+'/serie_'+str(i)+'.dat.gz'
        fileparam=directory+'/params_cbc_'+str(i)+'.dat.gz'
        print(fileserie)
        if i==0:
            filenoise=directories[3]+'/serie_'+str(i)+'.dat'
        else:
            filenoise=directories[3]+'/serie_'+str(i)+'.dat.gz'
        #Creating bbh ,noise and parameters dataframes . Finaldata will contain bbh+noise
        bbh1=pd.read_csv(fileserie,sep=' ', lineterminator='\n',header=None,names=['t0','ET1','ET2','ET3'],usecols=['t0','ET1'])
        noisedata=pd.read_csv(filenoise,sep=' ', lineterminator='\n',header=None,names=['t0','ET1','ET2','ET3'],usecols=['t0','ET1'])
        bbh1p=pd.read_csv(fileparam,sep=' ', lineterminator='\n',header=None,
                    names=['t0','tf','tmax','m1','m2','dist','ra','decl','psi','inc','phi0','type']) 
        a=bbh1p['tf'].values
        print(np.size(a))
        timeframes=pd.DataFrame(columns=['tf'])
        for j in range(0,np.size(a)):
            b=a[j].split('=')
            Tf=float(b[1])
            timeframes.at[j,'tf']=Tf  
        
        x=0
        nf=max(bbh1['t0'].values)
        #print(nf)
        ni=min(bbh1['t0'].values)+1
        c=timeframes['tf'].values
        #print(c)
        datalist=[]
        ndatalist=[]
        for sig_end in c[1:-1]:
            sig_end=float(sig_end)
            if (sig_end <= nf) & (sig_end >= ni) & (x<2047):
                sig_start = sig_end - 1
                data=bbh1[(sig_start<bbh1['t0']) & (bbh1['t0']<=sig_end)]
                del data['t0']
                datat=data.transpose()
                datat['Label']=1
                ldatat=np.array(datat)
                try:
                    datalist=np.vstack((datalist,ldatat))
                except:
                    datalist=ldatat
                x+=1 
        y=x
       # print(y)
        for l in range(int(ni),int(nf)+1):
            if x<=2*y-1:
                ndata=noisedata[(l<=noisedata['t0']) & (noisedata['t0']<l+1)]
                del ndata['t0']
                ndatat=ndata.transpose()
                ndatat['Label']=0
                lndatat=np.array(ndatat)
                try:
                    ndatalist=np.vstack((ndatalist,lndatat))
                except:
                    ndatalist=lndatat
                x+=1
        datalist=np.vstack((datalist,ndatalist))
        finaldata=pd.DataFrame(data=datalist)   
        sample_ind = [str(m) for m in range(4097)] 
        finaldata.columns=sample_ind
        finaldata=finaldata.rename(columns={"4096":"Label"})
        print(finaldata.shape)
        #print(k)
        if k==0:
            n=1
        elif k==1:
            n=4
        elif k==2:
            n=10
            
        outdir = '/home/marangio/Downloads/BBH1s'
        outname='new_BBH'+str(n)+'s_serie'+str(i)+'.csv.gz'
        fullname = os.path.join(outdir, outname) 
        finaldata.to_csv(fullname)